# Save and load an analysis

An `Analysis` is the frozen container a notebook holds: specs by role, the panel, the
posterior, evidence records, the assumption ledger, and provenance. `save_analysis` writes
the `analysis.axiom` directory format:

```
analysis.axiom/
├── manifest.json        format version, axiom version, hashes, declared bases and units
├── specs/<role>.json    one envelope per Spec
├── panel.csv            the Panel (17 significant digits) + panel_roles.json
├── posterior.npz        draws + JSON meta
└── evidence/            evidence.jsonl, ledger.jsonl
```

No pickle anywhere: loading replays specs by class name and rehydrates arrays.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from axiom.core import BASES, D, LedgerLine, Outcome, Posterior, TimeWindow, Treatment
from axiom.data import Panel, RoleMap, fit_scaling
from axiom.io import (
    FORMAT_VERSION,
    Analysis,
    ArtifactRegistry,
    FormatError,
    Provenance,
    environment_fingerprint,
    load_analysis,
    save_analysis,
)

In [ ]:
rng = np.random.default_rng(3)
df = pd.DataFrame({"u": np.repeat(["01", "02", "03"], 8), "t": np.tile(range(8), 3),
                   "y": rng.gamma(3, 10, 24), "x": rng.uniform(0, 100, 24)})
roles = RoleMap(unit="u", time="t",
                outcome=("y", Outcome(name="y_total", dimension=D.outcome)),
                treatments={"x": Treatment(name="x_dose", dimension=D.currency, unit="USD")})
panel = Panel(df, roles)
posterior = Posterior({"beta": rng.normal(0.5, 0.1, size=(2, 200))}, provenance={"seed": 3})

## Build the container

`Analysis` is immutable; `with_*` methods return a new one. `hashes()` is the set of content
hashes that becomes the provenance record.

In [ ]:
analysis = (
    Analysis(specs={"roles": roles, "scaling": fit_scaling(panel), "window": TimeWindow(start=0, stop=8)})
    .with_panel(panel)
    .with_posterior(posterior)
    .with_ledger_line(LedgerLine(kind="note", statement="toy analysis for the io notebook"))
)
print(analysis.summary())
for k, v in analysis.hashes().items():
    print(f"  {k:14s} {v[:16]}")

## Save

`save_analysis` refuses to overwrite unless told to, stamps provenance, and returns the
stamped `Analysis` — which compares equal to what `load_analysis` returns.

In [ ]:
root = Path(tempfile.mkdtemp()) / "toy.axiom"
saved = save_analysis(analysis, root, seed=3)
print(sorted(p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()))
print(FORMAT_VERSION, "|", saved.provenance.seed, saved.provenance.created)

In [ ]:
loaded = load_analysis(root)
print("equal after round-trip:", loaded == saved)
print(loaded.spec("window"), "|", loaded.panel, "|", loaded.posterior)
print(loaded.ledger[0].statement)

## Provenance and the environment

`Provenance` records the axiom version, a UTC timestamp, the content hashes, the seed, and the
environment fingerprint. You can build one explicitly (useful for deterministic tests).

In [ ]:
print(environment_fingerprint())
prov = Provenance(axiom_version="0.0.0", created="2026-08-21T00:00:00+00:00", hashes=analysis.hashes(), seed=3)
print(prov.content_hash()[:16])

## Declared bases travel with the analysis

If a domain declared extra base dimensions, the manifest records them and `load_analysis`
re-declares them before replaying specs (note 0002.6).

In [ ]:
import json

BASES.declare("mass", symbol="M")
root2 = root.parent / "with_mass.axiom"
save_analysis(Analysis(specs={"m": Treatment(name="seed_mass", dimension=D.mass, unit="kg")}), root2)
print(json.loads((root2 / "manifest.json").read_text())["bases"])

## Refusals

Overwriting without `overwrite=True`, a directory that is not an analysis, an unknown format
version, and a panel whose bytes no longer match their hash are all errors.

In [ ]:
try:
    save_analysis(analysis, root)
except FileExistsError as e:
    print("FileExistsError:", e)

m = root / "manifest.json"
m.write_text(m.read_text().replace(f'"format_version": "{FORMAT_VERSION}"', '"format_version": "99"'))
try:
    load_analysis(root)
except FormatError as e:
    print("FormatError:", e)

## A content-addressed registry

`ArtifactRegistry` stores specs by hash. `put` is idempotent; `get` verifies the hash on read.

In [ ]:
reg = ArtifactRegistry(root.parent / "registry")
h = reg.put(roles)
print(h[:16], reg.put(roles) == h, len(reg), h in reg)
print(reg.get(h) == roles)
print(list(reg))